# Traffic Sign Recognition - Training on Google Colab
# Huấn luyện nhận dạng biển báo giao thông

**Project:** Robot hỗ trợ người khiếm thị nhận diện biển báo giao thông  
**Dataset:** GTSRB (German Traffic Sign Recognition Benchmark)  
**Model:** MobileNetV2 (Transfer Learning)  
**Target Device:** ESP32-CAM  
**Date:** January 30, 2026

---

## Quy trình thực hiện:
1. **Phần 1:** Cài đặt môi trường (Setup Environment)
2. **Phần 2:** Tải và xử lý dataset (Dataset Preparation)
3. **Phần 3:** Huấn luyện mô hình (Model Training)
4. **Phần 4:** Tối ưu hóa và xuất mô hình (Model Export)
5. **Phần 5:** Đánh giá kết quả (Evaluation)

**Thời gian dự kiến:** ~60-90 phút  
**GPU:** T4 (miễn phí từ Google Colab)

---

## Lưu ý quan trọng:
- Chạy các cell **theo thứ tự từ trên xuống**
- Mỗi cell phải chạy thành công trước khi chuyển sang cell tiếp theo
- Dữ liệu sẽ được lưu vào Google Drive để tránh mất khi session hết hạn
- Session Colab có giới hạn 12 giờ - hãy lưu checkpoint thường xuyên

---
# PHẦN 1: CÀI ĐẶT MÔI TRƯỜNG
---

## Cell 1.1: Kiểm tra GPU
Verify GPU availability / Kiểm tra xem Colab có cấp GPU không

In [ ]:
# Kiểm tra GPU có sẵn không
# Check if GPU is available

import tensorflow as tf
import sys

print("Python version:", sys.version)
print("TensorFlow version:", tf.__version__)
print("\nGPU available:", tf.config.list_physical_devices('GPU'))

# Nếu không có GPU, hãy bật GPU trong Runtime → Change runtime type → GPU
if len(tf.config.list_physical_devices('GPU')) == 0:
    print("\n⚠️ WARNING: No GPU detected!")
    print("Please enable GPU: Runtime → Change runtime type → Hardware accelerator → GPU")
else:
    print("\n✅ GPU detected! Training will be fast.")

## Cell 1.2: Mount Google Drive
Kết nối Google Drive để lưu trữ dữ liệu và mô hình

In [ ]:
# Mount Google Drive
# Kết nối với Google Drive (bạn sẽ cần cấp quyền)

from google.colab import drive
drive.mount('/content/drive')

print("\n✅ Google Drive mounted successfully!")
print("Your files will be saved to: /content/drive/MyDrive/")

## Cell 1.3: Tạo cấu trúc thư mục
Create folder structure in Google Drive

In [ ]:
# Tạo thư mục trong Google Drive
# Create project folders in Google Drive

import os

# Định nghĩa đường dẫn
BASE_DIR = '/content/drive/MyDrive/TrafficSignRobot'
DATASET_DIR = f'{BASE_DIR}/dataset'
FILTERED_DIR = f'{BASE_DIR}/dataset_filtered'
TRAIN_DIR = f'{BASE_DIR}/data_train'
VAL_DIR = f'{BASE_DIR}/data_val'
TEST_DIR = f'{BASE_DIR}/data_test'
MODELS_DIR = f'{BASE_DIR}/models'

# Tạo thư mục
os.makedirs(BASE_DIR, exist_ok=True)
os.makedirs(DATASET_DIR, exist_ok=True)
os.makedirs(FILTERED_DIR, exist_ok=True)
os.makedirs(TRAIN_DIR, exist_ok=True)
os.makedirs(VAL_DIR, exist_ok=True)
os.makedirs(TEST_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

print("✅ Folder structure created:")
print(f"  - Base: {BASE_DIR}")
print(f"  - Dataset: {DATASET_DIR}")
print(f"  - Filtered: {FILTERED_DIR}")
print(f"  - Train: {TRAIN_DIR}")
print(f"  - Val: {VAL_DIR}")
print(f"  - Test: {TEST_DIR}")
print(f"  - Models: {MODELS_DIR}")

## Cell 1.4: Cài đặt thư viện
Install required libraries

In [ ]:
# Cài đặt các thư viện cần thiết
# Install required packages

!pip install -q kagglehub
!pip install -q pillow scikit-learn pandas matplotlib seaborn

print("\n✅ All packages installed successfully!")

# Import các thư viện
import kagglehub
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from pathlib import Path
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing.image import ImageDataGenerator

print("\n✅ All libraries imported!")

## Cell 1.5: Cài đặt Kaggle API
Setup Kaggle API credentials

In [ ]:
# Upload kaggle.json từ máy tính của bạn
# Upload kaggle.json file from your computer

from google.colab import files

print("📤 Please upload your kaggle.json file:")
print("   (Get it from: https://www.kaggle.com → Account → API → Create New Token)")
print("\nClick the 'Choose Files' button below...\n")

uploaded = files.upload()

# Cấu hình Kaggle API
!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

print("\n✅ Kaggle API configured successfully!")

---
# PHẦN 2: TẢI VÀ XỬ LÝ DATASET
---

## Cell 2.1: Tải GTSRB Dataset
Download GTSRB dataset from Kaggle (~1.2GB, 10-15 phút)

In [ ]:
# Tải GTSRB dataset từ Kaggle
# Download GTSRB dataset from Kaggle

import shutil

print("📥 Downloading GTSRB dataset from Kaggle...")
print("   Size: ~1.2GB | Time: 10-15 minutes\n")

# Tải dataset
path = kagglehub.dataset_download("meowmeowmeowmeowmeow/gtsrb-german-traffic-sign")
print(f"\n✅ Downloaded to: {path}")

# Copy sang Google Drive
print("\n📦 Copying to Google Drive...")
if os.path.exists(f"{path}/Train"):
    shutil.copytree(f"{path}/Train", f"{DATASET_DIR}/Train", dirs_exist_ok=True)
    print(f"✅ Training data copied to: {DATASET_DIR}/Train")

if os.path.exists(f"{path}/Test"):
    shutil.copytree(f"{path}/Test", f"{DATASET_DIR}/Test", dirs_exist_ok=True)
    print(f"✅ Test data copied to: {DATASET_DIR}/Test")

# Kiểm tra số lượng classes
train_path = Path(f"{DATASET_DIR}/Train")
num_classes = len(list(train_path.iterdir()))
print(f"\n✅ Dataset ready! Total classes: {num_classes}")

## Cell 2.2: Định nghĩa 15 lớp biển báo Việt Nam
Define 15 Vietnamese traffic sign classes

In [ ]:
# Mapping 15 biển báo GTSRB sang tên tiếng Việt
# Map 15 GTSRB classes to Vietnamese sign names

SELECTED_CLASSES = {
    0: "toc_do_20",           # Speed limit 20 km/h
    1: "toc_do_30",           # Speed limit 30 km/h
    2: "toc_do_50",           # Speed limit 50 km/h
    14: "dung",               # Stop sign
    17: "cam_di_vao",         # No entry
    25: "dang_thi_cong",      # Road work
    28: "tre_em_qua_duong",   # Children crossing
    31: "nguoi_di_bo",        # Pedestrian crossing
    33: "re_phai",            # Turn right ahead
    34: "re_trai",            # Turn left ahead
    35: "chi_di_thang",       # Ahead only
    38: "di_ben_phai",        # Keep right
    39: "di_ben_trai",        # Keep left
    40: "bung_binh",          # Roundabout mandatory
    41: "het_han_che"         # End of no passing
}

IMAGES_PER_CLASS = 200  # Số ảnh mỗi lớp

print("📋 Danh sách 15 biển báo đã chọn:")
print("="*60)
for gtsrb_id, vn_name in SELECTED_CLASSES.items():
    print(f"  Class {gtsrb_id:2d}: {vn_name}")
print("="*60)
print(f"\n✅ Total: {len(SELECTED_CLASSES)} classes")
print(f"✅ Target: {IMAGES_PER_CLASS} images per class")
print(f"✅ Total images: {len(SELECTED_CLASSES) * IMAGES_PER_CLASS}")

## Cell 2.3: Lọc và sao chép 15 lớp
Filter and copy 15 selected classes

In [ ]:
# Lọc 15 lớp từ 43 lớp GTSRB
# Filter 15 classes from 43 GTSRB classes

import random

print("🔍 Filtering 15 Vietnamese traffic sign classes...\n")

source_dir = Path(f"{DATASET_DIR}/Train")
target_dir = Path(FILTERED_DIR)
RANDOM_SEED = 42
rng = random.Random(RANDOM_SEED)

total_copied = 0

for class_id, class_name in SELECTED_CLASSES.items():
    src_candidates = [source_dir / f"{class_id:05d}", source_dir / str(class_id)]
    src_class_dir = next((p for p in src_candidates if p.exists()), src_candidates[0])
    dst_class_dir = target_dir / class_name
    
    if not src_class_dir.exists():
        print(f"⚠️  Class {class_id} not found, skipping...")
        continue
    
    # Lấy tất cả ảnh PPM/PNG
    images = sorted(src_class_dir.glob("*.ppm"))
    if not images:
        images = sorted(src_class_dir.glob("*.png"))

    # Random sample để reproducible
    if len(images) <= IMAGES_PER_CLASS:
        images_to_copy = images
    else:
        images_to_copy = sorted(rng.sample(images, IMAGES_PER_CLASS), key=lambda p: p.name)
    
    # Tạo thư mục đích
    dst_class_dir.mkdir(parents=True, exist_ok=True)
    
    # Sao chép ảnh
    for img_path in images_to_copy:
        shutil.copy2(img_path, dst_class_dir / img_path.name)
    
    copied_count = len(images_to_copy)
    total_copied += copied_count
    
    print(f"✓ Class {class_id:2d} ({class_name:20s}): {copied_count:3d} images")

print("\n" + "="*60)
print(f"✅ Total images copied: {total_copied}")
print(f"✅ Classes: {len(SELECTED_CLASSES)}")
print(f"✅ Average per class: {total_copied / len(SELECTED_CLASSES):.1f}")
print("="*60)

## Cell 2.4: Chuyển đổi PPM → JPEG 96×96
Convert PPM images to JPEG and resize to 96×96

In [ ]:
# Chuyển đổi PPM/PNG sang JPEG và resize về 96×96
# Convert PPM/PNG to JPEG and resize to 96×96

from PIL import Image
from tqdm import tqdm

TARGET_SIZE = (96, 96)
JPEG_QUALITY = 95

print(f"🖼️  Converting images → JPEG {TARGET_SIZE[0]}×{TARGET_SIZE[1]}...\n")

filtered_dir = Path(FILTERED_DIR)
image_files = sorted(filtered_dir.rglob("*.ppm")) + sorted(filtered_dir.rglob("*.png"))

print(f"Found {len(image_files)} source images to convert\n")

converted_count = 0
error_count = 0

for src_path in tqdm(image_files, desc="Converting"):
    try:
        # Mở ảnh nguồn
        img = Image.open(src_path)
        
        # Chuyển sang RGB và resize
        img = img.convert("RGB")
        img = img.resize(TARGET_SIZE, Image.Resampling.LANCZOS)
        
        # Lưu thành JPEG
        jpg_path = src_path.with_suffix(".jpg")
        img.save(jpg_path, "JPEG", quality=JPEG_QUALITY)
        
        # Xóa file nguồn gốc
        src_path.unlink()
        
        converted_count += 1
        
    except Exception as e:
        print(f"\n⚠️  Error converting {src_path.name}: {e}")
        error_count += 1

print("\n" + "="*60)
print(f"✅ Successfully converted: {converted_count} images")
if error_count > 0:
    print(f"⚠️  Errors: {error_count} images")
print("="*60)

## Cell 2.5: Chia dataset thành Train/Val/Test (70/15/15)
Split dataset into Train/Val/Test (70/15/15)

In [ ]:
# Chia dataset thành train/val/test (70/15/15) theo scene-aware grouping
# Split dataset into train/val/test (70/15/15) with scene-aware grouping

import random

TRAIN_RATIO = 0.7
VAL_RATIO = 0.15
TEST_RATIO = 0.15
RANDOM_SEED = 42

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print(f"📊 Splitting dataset: Train ({TRAIN_RATIO*100:.0f}%) / Val ({VAL_RATIO*100:.0f}%) / Test ({TEST_RATIO*100:.0f}%)\n")

filtered_dir = Path(FILTERED_DIR)
train_dir = Path(TRAIN_DIR)
val_dir = Path(VAL_DIR)
test_dir = Path(TEST_DIR)

# Reset output dirs
for d in [train_dir, val_dir, test_dir]:
    if d.exists():
        shutil.rmtree(d)
    d.mkdir(parents=True, exist_ok=True)

rng = random.Random(RANDOM_SEED)

def group_by_scene(paths):
    groups = {}
    for p in paths:
        stem = p.stem
        scene_id = stem.rsplit('_', 1)[0] if '_' in stem else stem
        groups.setdefault(scene_id, []).append(p)
    return groups

def split_scene_ids_by_image_count(groups, rng):
    scene_items = [(scene_id, len(paths)) for scene_id, paths in groups.items()]
    rng.shuffle(scene_items)
    scene_items.sort(key=lambda item: item[1], reverse=True)

    total_images = sum(size for _, size in scene_items)
    target_train = max(1, int(round(total_images * TRAIN_RATIO)))
    target_val = max(1, int(round(total_images * VAL_RATIO)))
    target_test = max(1, total_images - target_train - target_val)

    while target_train + target_val + target_test > total_images:
        if target_train >= target_val and target_train >= target_test and target_train > 1:
            target_train -= 1
        elif target_val >= target_test and target_val > 1:
            target_val -= 1
        elif target_test > 1:
            target_test -= 1
        else:
            break

    targets = {"train": target_train, "val": target_val, "test": target_test}
    counts = {"train": 0, "val": 0, "test": 0}
    split_ids = {"train": set(), "val": set(), "test": set()}

    # Seed: đảm bảo mỗi split có ít nhất 1 scene
    for split_name, (scene_id, size) in zip(["train", "val", "test"], scene_items[:3]):
        split_ids[split_name].add(scene_id)
        counts[split_name] += size

    # Greedy by deficit
    for scene_id, size in scene_items[3:]:
        deficits = {name: targets[name] - counts[name] for name in ["train", "val", "test"]}
        max_deficit = max(deficits.values())
        candidates = [name for name, deficit in deficits.items() if deficit == max_deficit]
        chosen = rng.choice(candidates)
        split_ids[chosen].add(scene_id)
        counts[chosen] += size

    return split_ids["train"], split_ids["val"], split_ids["test"]

total_train = 0
total_val = 0
total_test = 0

for class_dir in sorted(filtered_dir.iterdir()):
    if not class_dir.is_dir():
        continue

    class_name = class_dir.name
    images = sorted(class_dir.glob("*.jpg"))

    if not images:
        print(f"⚠️  No JPEG images in {class_name}")
        continue

    groups = group_by_scene(images)
    scene_ids = list(groups.keys())

    if len(scene_ids) < 3:
        shuffled = images[:]
        rng.shuffle(shuffled)
        n = len(shuffled)
        n_train = max(1, int(round(n * TRAIN_RATIO)))
        n_val = max(1, int(round(n * VAL_RATIO)))
        n_test = n - n_train - n_val
        if n_test <= 0:
            n_test = 1
            if n_train > n_val and n_train > 1:
                n_train -= 1
            elif n_val > 1:
                n_val -= 1

        train_images = shuffled[:n_train]
        val_images = shuffled[n_train:n_train + n_val]
        test_images = shuffled[n_train + n_val:]
    else:
        train_ids, val_ids, test_ids = split_scene_ids_by_image_count(groups, rng)
        train_images = [p for sid in train_ids for p in groups[sid]]
        val_images = [p for sid in val_ids for p in groups[sid]]
        test_images = [p for sid in test_ids for p in groups[sid]]

    train_images = sorted(train_images)
    val_images = sorted(val_images)
    test_images = sorted(test_images)

    train_class_dir = train_dir / class_name
    val_class_dir = val_dir / class_name
    test_class_dir = test_dir / class_name

    train_class_dir.mkdir(parents=True, exist_ok=True)
    val_class_dir.mkdir(parents=True, exist_ok=True)
    test_class_dir.mkdir(parents=True, exist_ok=True)

    for img_path in train_images:
        shutil.copy2(img_path, train_class_dir / img_path.name)
    for img_path in val_images:
        shutil.copy2(img_path, val_class_dir / img_path.name)
    for img_path in test_images:
        shutil.copy2(img_path, test_class_dir / img_path.name)

    total_train += len(train_images)
    total_val += len(val_images)
    total_test += len(test_images)

    print(
        f"✓ {class_name:20s}: train={len(train_images):3d}, "
        f"val={len(val_images):3d}, test={len(test_images):3d}"
    )

print("\n" + "="*60)
print(f"✅ Total train images: {total_train}")
print(f"✅ Total val images:   {total_val}")
print(f"✅ Total test images:  {total_test}")
print(f"✅ Grand total:        {total_train + total_val + total_test}")
print("="*60)

## Cell 2.6: Hiển thị mẫu ảnh
Display sample images from each class

In [ ]:
# Hiển thị mẫu ảnh từ mỗi lớp
# Display sample images from each class

import matplotlib.pyplot as plt

train_dir = Path(TRAIN_DIR)
class_dirs = sorted([d for d in train_dir.iterdir() if d.is_dir()])

# Tạo grid 3×5 (15 classes)
fig, axes = plt.subplots(3, 5, figsize=(15, 9))
fig.suptitle('Sample Images from Each Class', fontsize=16, fontweight='bold')

for idx, class_dir in enumerate(class_dirs):
    row = idx // 5
    col = idx % 5
    
    # Lấy ảnh đầu tiên
    sample_image = list(class_dir.glob("*.jpg"))[0]
    img = Image.open(sample_image)
    
    # Hiển thị
    axes[row, col].imshow(img)
    axes[row, col].set_title(class_dir.name, fontsize=10)
    axes[row, col].axis('off')

plt.tight_layout()
plt.savefig(f"{MODELS_DIR}/sample_images.png", dpi=150, bbox_inches='tight')
plt.show()

print("\n✅ Sample images saved to:", f"{MODELS_DIR}/sample_images.png")

---
# PHẦN 3: HUẤN LUYỆN MÔ HÌNH
---

## Cell 3.1: Thiết lập Data Generators
Setup data generators with augmentation

In [ ]:
# Thiết lập data generators với augmentation
# Setup data generators with augmentation

from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMG_SIZE = (96, 96)
BATCH_SIZE = 32

# Training data generator (có augmentation)
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    brightness_range=[0.7, 1.3],
    zoom_range=0.2,
    horizontal_flip=False
)

# Validation/Test data generator (không augmentation)
eval_datagen = ImageDataGenerator(rescale=1./255)

# Load data
train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

val_generator = eval_datagen.flow_from_directory(
    VAL_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

test_generator = eval_datagen.flow_from_directory(
    TEST_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False  # Không shuffle để đánh giá chính xác
)

print("\n" + "="*60)
print(f"✅ Training samples:   {train_generator.samples}")
print(f"✅ Validation samples: {val_generator.samples}")
print(f"✅ Test samples:       {test_generator.samples}")
print(f"✅ Number of classes: {train_generator.num_classes}")
print(f"✅ Batch size: {BATCH_SIZE}")
print("="*60)

# Lưu class labels
class_labels = list(train_generator.class_indices.keys())
print("\nClass labels:", class_labels)

## Cell 3.2: Xây dựng mô hình MobileNetV2
Build MobileNetV2 model with transfer learning

In [ ]:
# Xây dựng mô hình MobileNetV2
# Build MobileNetV2 model

from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models

# Tham số mô hình
IMG_SHAPE = (96, 96, 3)
NUM_CLASSES = train_generator.num_classes
ALPHA = 0.35  # Width multiplier (càng nhỏ càng nhẹ)

print("🏗️  Building MobileNetV2 model...\n")

# Load MobileNetV2 base model (pretrained on ImageNet)
base_model = MobileNetV2(
    input_shape=IMG_SHAPE,
    alpha=ALPHA,
    include_top=False,
    weights='imagenet'
)

# Freeze base model (transfer learning)
base_model.trainable = False

# Xây dựng mô hình hoàn chỉnh
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.3),
    layers.Dense(NUM_CLASSES, activation='softmax')
])

# Compile model
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("✅ Model built successfully!\n")
model.summary()

print("\n" + "="*60)
print(f"✅ Input shape: {IMG_SHAPE}")
print(f"✅ Number of classes: {NUM_CLASSES}")
print(f"✅ MobileNetV2 alpha: {ALPHA}")
print(f"✅ Base model trainable: {base_model.trainable}")
print("="*60)

## Cell 3.3: Huấn luyện mô hình
Train the model (50 epochs, ~20-30 minutes on GPU)

In [ ]:
# Huấn luyện mô hình
# Train the model

EPOCHS = 50

print(f"🚀 Starting training for {EPOCHS} epochs...\n")
print("This will take approximately 20-30 minutes on GPU T4\n")

# Callbacks
callbacks = [
    # Early stopping (dừng sớm nếu không cải thiện)
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy',
        patience=10,
        restore_best_weights=True,
        verbose=1
    ),
    
    # Model checkpoint (lưu model tốt nhất)
    tf.keras.callbacks.ModelCheckpoint(
        filepath=f"{MODELS_DIR}/best_model.h5",
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    ),
    
    # Reduce learning rate on plateau
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-7,
        verbose=1
    )
]

# Bắt đầu huấn luyện (validation dùng val split, test chỉ để đánh giá cuối)
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)

print("\n" + "="*60)
print("✅ Training complete!")
print(f"✅ Best model saved to: {MODELS_DIR}/best_model.h5")
print("="*60)

## Cell 3.4: Vẽ biểu đồ huấn luyện
Plot training curves (accuracy and loss)

In [ ]:
# Vẽ biểu đồ quá trình huấn luyện
# Plot training history

import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy plot
ax1.plot(history.history['accuracy'], label='Train Accuracy', linewidth=2)
ax1.plot(history.history['val_accuracy'], label='Validation Accuracy', linewidth=2)
ax1.set_title('Model Accuracy', fontsize=14, fontweight='bold')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Loss plot
ax2.plot(history.history['loss'], label='Train Loss', linewidth=2)
ax2.plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
ax2.set_title('Model Loss', fontsize=14, fontweight='bold')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f"{MODELS_DIR}/training_curves.png", dpi=150, bbox_inches='tight')
plt.show()

# In kết quả cuối cùng
final_train_acc = history.history['accuracy'][-1]
final_val_acc = history.history['val_accuracy'][-1]

print("\n" + "="*60)
print(f"✅ Final Training Accuracy:   {final_train_acc*100:.2f}%")
print(f"✅ Final Validation Accuracy: {final_val_acc*100:.2f}%")
print(f"✅ Training curves saved to: {MODELS_DIR}/training_curves.png")
print("="*60)

---
# PHẦN 4: TỐI ƯU HÓA VÀ XUẤT MÔ HÌNH
---

## Cell 4.1: Chuyển đổi sang TFLite
Convert to TensorFlow Lite format

In [ ]:
# Chuyển đổi mô hình sang TFLite
# Convert model to TensorFlow Lite

print("🔄 Converting model to TFLite format...\n")

# Load best model
best_model = tf.keras.models.load_model(f"{MODELS_DIR}/best_model.h5")

# Convert to TFLite
converter = tf.lite.TFLiteConverter.from_keras_model(best_model)
tflite_model = converter.convert()

# Lưu TFLite model (float32)
tflite_path = f"{MODELS_DIR}/traffic_sign_model_float32.tflite"
with open(tflite_path, 'wb') as f:
    f.write(tflite_model)

model_size_kb = len(tflite_model) / 1024

print("✅ TFLite conversion complete!")
print(f"✅ Model saved to: {tflite_path}")
print(f"✅ Model size (float32): {model_size_kb:.2f} KB")
print("\n⚠️  Note: This is float32 model. We'll quantize to int8 next for ESP32.")

## Cell 4.2: int8 Quantization
Apply int8 quantization to reduce size (~4× smaller)

In [ ]:
# int8 Quantization (giảm kích thước ~4 lần)
# int8 Quantization (reduce size ~4×)

print("⚙️  Applying int8 quantization...\n")

# Representative dataset cho quantization
def representative_dataset():
    for _ in range(100):
        data = next(iter(train_generator))[0]
        yield [data.astype(np.float32)]

# Convert với int8 quantization
converter = tf.lite.TFLiteConverter.from_keras_model(best_model)

# Optimization settings
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.uint8
converter.inference_output_type = tf.uint8

# Convert
tflite_quant_model = converter.convert()

# Lưu int8 model
tflite_quant_path = f"{MODELS_DIR}/traffic_sign_model_int8.tflite"
with open(tflite_quant_path, 'wb') as f:
    f.write(tflite_quant_model)

model_size_kb = len(tflite_quant_model) / 1024

print("\n" + "="*60)
print("✅ int8 quantization complete!")
print(f"✅ Model saved to: {tflite_quant_path}")
print(f"✅ Model size (int8): {model_size_kb:.2f} KB")
print("="*60)

if model_size_kb < 500:
    print("\n✅ Model size is suitable for ESP32-CAM (< 500KB)")
else:
    print("\n⚠️  Warning: Model size > 500KB. Consider using smaller alpha (0.2)")

## Cell 4.3: Tạo C++ Header File
Generate C++ header file for Arduino

In [ ]:
# Tạo C++ header file cho Arduino
# Generate C++ header file for Arduino

print("📝 Generating C++ header file for Arduino...\n")

def tflite_to_c_array(tflite_path, output_path):
    """Convert TFLite model to C array"""
    with open(tflite_path, 'rb') as f:
        tflite_binary = f.read()
    
    # Convert to hex array
    hex_array = ',\n  '.join(
        ', '.join(f'0x{byte:02x}' for byte in tflite_binary[i:i+12])
        for i in range(0, len(tflite_binary), 12)
    )
    
    # Generate C header
    c_header = f"""// Auto-generated TFLite model for ESP32-CAM
// Traffic Sign Recognition Model
// Generated: January 30, 2026
// Model size: {len(tflite_binary)} bytes

#ifndef MODEL_DATA_H
#define MODEL_DATA_H

// Align to 8 bytes for better performance
alignas(8) const unsigned char model_data[] = {{
  {hex_array}
}};

const unsigned int model_data_len = {len(tflite_binary)};

#endif  // MODEL_DATA_H
"""
    
    with open(output_path, 'w') as f:
        f.write(c_header)
    
    return len(tflite_binary)

# Generate header file
h_file_path = f"{MODELS_DIR}/model_data.h"
model_size = tflite_to_c_array(tflite_quant_path, h_file_path)

print("✅ C++ header file generated!")
print(f"✅ File saved to: {h_file_path}")
print(f"✅ Array size: {model_size} bytes ({model_size/1024:.2f} KB)")

# Tạo file class labels
labels_path = f"{MODELS_DIR}/class_labels.txt"
with open(labels_path, 'w', encoding='utf-8') as f:
    for label in class_labels:
        f.write(f"{label}\n")

print(f"\n✅ Class labels saved to: {labels_path}")
print("\nClass labels:")
for i, label in enumerate(class_labels):
    print(f"  {i}: {label}")

## Cell 4.4: Tải xuống model files
Download model files to your computer

In [ ]:
# Tải xuống model files về máy tính
# Download model files to your computer

from google.colab import files

print("📥 Preparing files for download...\n")

files_to_download = [
    (f"{MODELS_DIR}/traffic_sign_model_int8.tflite", "TFLite int8 model"),
    (f"{MODELS_DIR}/model_data.h", "C++ header for Arduino"),
    (f"{MODELS_DIR}/class_labels.txt", "Class labels"),
    (f"{MODELS_DIR}/best_model.h5", "Best Keras model (backup)"),
    (f"{MODELS_DIR}/training_curves.png", "Training curves"),
    (f"{MODELS_DIR}/sample_images.png", "Sample images")
]

print("Available files for download:")
for i, (path, desc) in enumerate(files_to_download, 1):
    if os.path.exists(path):
        size = os.path.getsize(path) / 1024
        print(f"  {i}. {desc}: {size:.2f} KB")

print("\n" + "="*60)
print("To download files, run the cell below.")
print("Or access them directly from Google Drive:")
print(f"  {MODELS_DIR}")
print("="*60)

In [ ]:
# Uncomment để tải file xuống
# Uncomment to download files

# Download model_data.h (quan trọng nhất cho ESP32)
# files.download(f"{MODELS_DIR}/model_data.h")

# Download TFLite model
# files.download(f"{MODELS_DIR}/traffic_sign_model_int8.tflite")

# Download class labels
# files.download(f"{MODELS_DIR}/class_labels.txt")

print("Bỏ dấu # ở dòng muốn tải và chạy lại cell này")
print("Uncomment the lines above and run this cell to download")

---
# PHẦN 5: ĐÁNH GIÁ KẾT QUẢ
---

## Cell 5.1: Đánh giá trên test set
Evaluate model on test set

In [ ]:
# Đánh giá mô hình trên test set
# Evaluate model on test set

print("📊 Evaluating model on test set...\n")

# Load best model
best_model = tf.keras.models.load_model(f"{MODELS_DIR}/best_model.h5")

# Evaluate
test_loss, test_accuracy = best_model.evaluate(test_generator, verbose=1)

print("\n" + "="*60)
print(f"✅ Test Loss: {test_loss:.4f}")
print(f"✅ Test Accuracy: {test_accuracy*100:.2f}%")
print("="*60)

if test_accuracy >= 0.90:
    print("\n✅ EXCELLENT! Model meets target accuracy (>90%)")
elif test_accuracy >= 0.85:
    print("\n⚠️  Good, but below target. Consider training longer or adjusting hyperparameters.")
else:
    print("\n❌ Below 85%. Model needs improvement.")

## Cell 5.2: Ma trận nhầm lẫn (Confusion Matrix)
Generate confusion matrix to analyze misclassifications

In [ ]:
# Ma trận nhầm lẫn (Confusion Matrix)
# Confusion Matrix

from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

print("📈 Generating confusion matrix...\n")

# Dự đoán trên test set
test_generator.reset()
predictions = best_model.predict(test_generator, verbose=1)
y_pred = np.argmax(predictions, axis=1)
y_true = test_generator.classes

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)

# Plot
plt.figure(figsize=(12, 10))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=class_labels,
    yticklabels=class_labels,
    cbar_kws={'label': 'Count'}
)
plt.title('Confusion Matrix', fontsize=16, fontweight='bold')
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(f"{MODELS_DIR}/confusion_matrix.png", dpi=150, bbox_inches='tight')
plt.show()

print(f"\n✅ Confusion matrix saved to: {MODELS_DIR}/confusion_matrix.png")

## Cell 5.3: Classification Report
Detailed metrics per class (precision, recall, f1-score)

In [ ]:
# Báo cáo phân loại chi tiết
# Detailed classification report

print("📋 Classification Report:\n")

report = classification_report(
    y_true,
    y_pred,
    target_names=class_labels,
    digits=4
)

print(report)

# Lưu report
report_path = f"{MODELS_DIR}/classification_report.txt"
with open(report_path, 'w') as f:
    f.write(report)

print(f"\n✅ Report saved to: {report_path}")

## Cell 5.4: Kiểm tra kích thước mô hình TFLite
Verify TFLite model accuracy (int8 vs float32)

In [ ]:
# Kiểm tra độ chính xác của mô hình TFLite int8
# Verify int8 TFLite model accuracy

print("🔍 Testing int8 TFLite model accuracy...\n")

# Load TFLite model
interpreter = tf.lite.Interpreter(model_path=f"{MODELS_DIR}/traffic_sign_model_int8.tflite")
interpreter.allocate_tensors()

# Get input/output details
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print("Input details:")
print(f"  Shape: {input_details[0]['shape']}")
print(f"  Type: {input_details[0]['dtype']}")

print("\nOutput details:")
print(f"  Shape: {output_details[0]['shape']}")
print(f"  Type: {output_details[0]['dtype']}")

# Test trên 100 ảnh đầu tiên
test_generator.reset()
correct = 0
total = 100

print(f"\nTesting on {total} images...")

for i in range(total):
    data, labels = next(test_generator)
    
    # Preprocess input (convert to uint8)
    input_data = (data[0] * 255).astype(np.uint8)
    input_data = np.expand_dims(input_data, axis=0)
    
    # Run inference
    interpreter.set_tensor(input_details[0]['index'], input_data)
    interpreter.invoke()
    output = interpreter.get_tensor(output_details[0]['index'])
    
    # Compare prediction
    if np.argmax(output) == np.argmax(labels[0]):
        correct += 1

tflite_accuracy = correct / total

print("\n" + "="*60)
print(f"✅ TFLite int8 accuracy (sample): {tflite_accuracy*100:.2f}%")
print(f"✅ Keras model accuracy (full):  {test_accuracy*100:.2f}%")
print(f"✅ Accuracy drop due to quantization: {(test_accuracy - tflite_accuracy)*100:.2f}%")
print("="*60)

if abs(test_accuracy - tflite_accuracy) < 0.05:
    print("\n✅ Quantization successful! Accuracy drop < 5%")
else:
    print("\n⚠️  Significant accuracy drop. May need more representative data for quantization.")

---
# KẾT THÚC - SUMMARY
---

## Cell 6.1: Tóm tắt kết quả
Final summary and next steps

In [ ]:
# Tóm tắt kết quả cuối cùng
# Final summary

print("\n" + "="*70)
print(" "*20 + "TRAINING COMPLETE - TÓM TẮT KẾT QUẢ")
print("="*70)

# Dataset info
print("\n📊 DATASET:")
print(f"  ✓ Source: GTSRB (German Traffic Sign Recognition Benchmark)")
print(f"  ✓ Classes: {len(SELECTED_CLASSES)} Vietnamese traffic signs")
print(f"  ✓ Training images:   {train_generator.samples}")
print(f"  ✓ Validation images: {val_generator.samples}")
print(f"  ✓ Test images:       {test_generator.samples}")
print(f"  ✓ Image size: 96×96 RGB")

# Model info
print("\n🏗️  MODEL:")
print(f"  ✓ Architecture: MobileNetV2 (alpha={ALPHA})")
print(f"  ✓ Transfer learning: ImageNet pretrained")
print(f"  ✓ Training epochs: {len(history.history['accuracy'])}")
print(f"  ✓ Final train accuracy: {history.history['accuracy'][-1]*100:.2f}%")
print(f"  ✓ Final validation accuracy: {history.history['val_accuracy'][-1]*100:.2f}%")

# Performance
print("\n📈 PERFORMANCE:")
print(f"  ✓ Test accuracy: {test_accuracy*100:.2f}%")
print(f"  ✓ Test loss: {test_loss:.4f}")

# Model sizes
h5_size = os.path.getsize(f"{MODELS_DIR}/best_model.h5") / 1024
tflite_size = os.path.getsize(f"{MODELS_DIR}/traffic_sign_model_int8.tflite") / 1024

print("\n💾 MODEL SIZES:")
print(f"  ✓ Keras (.h5): {h5_size:.2f} KB")
print(f"  ✓ TFLite (int8): {tflite_size:.2f} KB")
print(f"  ✓ Size reduction: {(1 - tflite_size/h5_size)*100:.1f}%")

# Files generated
print("\n📁 FILES GENERATED:")
print(f"  ✓ {MODELS_DIR}/best_model.h5")
print(f"  ✓ {MODELS_DIR}/traffic_sign_model_int8.tflite")
print(f"  ✓ {MODELS_DIR}/model_data.h (for Arduino)")
print(f"  ✓ {MODELS_DIR}/class_labels.txt")
print(f"  ✓ {MODELS_DIR}/training_curves.png")
print(f"  ✓ {MODELS_DIR}/confusion_matrix.png")
print(f"  ✓ {MODELS_DIR}/classification_report.txt")

# Next steps
print("\n🚀 NEXT STEPS:")
print("  1. Download model_data.h from Google Drive")
print("  2. Copy to your Arduino project folder")
print("  3. Install TensorFlow Lite Micro library in Arduino IDE")
print("  4. Integrate with ESP32-CAM code (Phase 3)")
print("  5. Test inference on real traffic sign images")

print("\n" + "="*70)
print(" "*25 + "✅ TRAINING SUCCESSFUL!")
print("="*70 + "\n")

# Check if meets requirements
if test_accuracy >= 0.90 and tflite_size < 500:
    print("🎉 CONGRATULATIONS! Model meets all requirements:")
    print("   ✅ Accuracy > 90%")
    print("   ✅ Model size < 500KB")
    print("   ✅ Ready for ESP32-CAM deployment!")
else:
    if test_accuracy < 0.90:
        print("⚠️  Accuracy below 90%. Consider:")
        print("   - Training for more epochs")
        print("   - Using larger alpha (0.5 or 0.75)")
        print("   - Adding more training data")
    if tflite_size >= 500:
        print("⚠️  Model size >= 500KB. Consider:")
        print("   - Using smaller alpha (0.2)")
        print("   - Reducing image size to 64×64")